# NB00 setup
Checks GPU and versions, installs espeak-ng, validates `backend/words.json` against the wav2vec2 phoneme vocabulary,
and writes a data inventory. No metrics here, only counts of what exists.

Kaggle settings: Internet ON. Accelerator: none is fine.

In [ ]:
# Repo code (per.py, words.json). Kaggle cannot see your laptop, so pick one:
#   A) public GitHub repo + Internet ON in notebook settings -> git clone below
#   B) private repo -> upload the repo folder as a private Kaggle Dataset "spellspeak-code" and add it here
import json, os, platform, subprocess, sys, time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

REPO_URL = "https://github.com/ddrlve/SpellSpeak.git"
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
CANDIDATES = [Path("/kaggle/input/spellspeak-code"), Path.cwd().parent, WORK / "SpellSpeak"]
REPO = next((p for p in CANDIDATES if (p / "backend" / "per.py").exists()), None)
if REPO is None:
    REPO = WORK / "SpellSpeak"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO / "backend"))
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO, "| results:", RESULTS)

In [ ]:
def cpu_name() -> str:
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def device_info(force_cpu: bool = False) -> dict:
    name, threads = cpu_name(), os.cpu_count()
    try:
        import torch
        threads = torch.get_num_threads()
        if torch.cuda.is_available() and not force_cpu:
            name = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return {"name": name, "threads": threads, "platform": platform.platform()}


def versions(*pkgs: str) -> dict:
    out = {}
    for p in pkgs:
        try:
            out[p] = metadata.version(p)
        except metadata.PackageNotFoundError:
            out[p] = None
    return out


def write_result(notebook: str, data: dict, metrics: dict, notes: str, pkgs: tuple = ()) -> Path:
    """Shape from docs/NOTEBOOKS.md. Only values computed in this run go into metrics."""
    out = {"notebook": notebook, "date": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "device": device_info(), "data": data, "metrics": metrics, "notes": notes,
           "versions": versions(*pkgs)}
    path = RESULTS / f"{notebook}.json"
    path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(out, indent=2, ensure_ascii=False))
    return path

In [ ]:
!nvidia-smi || echo "no GPU"
!apt-get -qq update && apt-get -qq install -y espeak-ng > /dev/null && espeak-ng --version
!pip install -q phonemizer

In [ ]:
from phonemizer import phonemize
from phonemizer.separator import Separator

_ipa_cache: dict[str, list[str]] = {}


def target_ipa(word: str) -> list[str]:
    """Same call as backend/server.py so the notebook scores exactly what the app scores."""
    if word not in _ipa_cache:
        out = phonemize(word, language="en-us", backend="espeak",
                        separator=Separator(phone=" ", word="", syllable=""), strip=True)
        _ipa_cache[word] = out.split()
    return _ipa_cache[word]

In [ ]:
# Validate candidate words: does phonemizer produce tokens, and are all tokens in the model vocabulary?
from transformers import Wav2Vec2Processor

MODEL = "facebook/wav2vec2-lv-60-espeak-cv-ft"
VOCAB = set(Wav2Vec2Processor.from_pretrained(MODEL).tokenizer.get_vocab())
words = json.loads((REPO / "backend" / "words.json").read_text(encoding="utf-8"))["words"]
rows = []
for w in words:
    ipa = target_ipa(w["word"])
    oov = [p for p in ipa if p not in VOCAB]
    rows.append({"word": w["word"], "target": w["target"], "tier": w["tier"], "ipa": " ".join(ipa),
                 "oov_tokens": " ".join(oov), "usable": bool(ipa) and not oov})
import pandas as pd
words_df = pd.DataFrame(rows)
words_df

In [ ]:
# Data inventory: which Kaggle inputs are attached, and what own clips exist.
inputs = sorted(p.name for p in Path("/kaggle/input").iterdir()) if Path("/kaggle/input").exists() else []
own_meta = Path("/kaggle/input/spellspeak-own-clips/metadata.csv")
own = {"attached": own_meta.exists()}
if own_meta.exists():
    m = pd.read_csv(own_meta)
    own.update(n_clips=len(m), speakers=int(m.speaker_id.nunique()), words=int(m.word.nunique()),
               missing_files=int(sum(not (own_meta.parent / f).exists() for f in m.file)))
print("inputs:", inputs)
print("own clips:", own)

In [ ]:
write_result(
    "NB00_setup",
    data={"source": "inventory", "kaggle_inputs": inputs, "own_clips": own,
          "n_utterances": own.get("n_clips", 0), "speakers": own.get("speakers", 0)},
    metrics={},
    notes="Inventory only. words.json validation: " + json.dumps(
        {"usable": words_df[words_df.usable].word.tolist(),
         "not_usable": words_df[~words_df.usable][["word", "oov_tokens"]].to_dict("records")}, ensure_ascii=False),
    pkgs=("torch", "transformers", "phonemizer"),
)
words_df.to_csv(RESULTS / "NB00_words_validation.csv", index=False)